# Inference numerical boundaries

These deterministic examples compare installed original PyCBC calculations with JAX. Reference assertions compare dtype, shape and bytes; defaults retain device execution. No saved data files are required. The final physical comparisons use complex128 data, matching the original Gaussian model's precision contract. Single-precision evidence is shown at supported Array and relative-kernel boundaries.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import sys
import numpy as np
import jax
import jax.numpy as jnp
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array, FrequencySeries
from pycbc.inference.models import tools, tools_jax, relbin_jax, relbin_cpu
from pycbc.waveform.generator_jax import _arrival_time_and_shift
jax.config.update("jax_enable_x64", True)

def bits(value):
    a = np.asarray(value)
    return a.dtype, a.shape, a.tobytes()

def exact(actual, original):
    assert bits(actual) == bits(original)

print({"Python": sys.version.split()[0], "NumPy": np.__version__,
       "JAX": jax.__version__, "devices": [d.device_kind for d in jax.devices()]})

{'Python': '3.12.7', 'NumPy': '2.5.2', 'JAX': '0.11.1', 'devices': ['cpu']}


## Product precision

The original single-precision inner product rounds products before its double-precision accumulation. JAX promotes inputs first. A single element proves this is a product boundary, independent of reduction ordering. At physical complex64 amplitudes, the same distinction can produce underflow.

In [2]:
for values in (np.asarray([1.234567], np.float32), np.asarray([1e-23+2e-23j], np.complex64)):
    with CPUScheme():
        original = Array(values).inner(Array(values))
    with JAXScheme():
        device = jnp.asarray(values)
        default = tools_jax.inner(device, device)
    with JAXScheme(reference_operations=("sum",)):
        reduction_only = tools_jax.inner(device, device)
    with JAXScheme(reference_operations=("inference_inner",)):
        reference = tools_jax.inner(device, device)
    exact(reference, original)
    exact(reduction_only, default)
    print({"dtype": str(values.dtype), "original": original,
           "JAX": np.asarray(default).item(), "reference": np.asarray(reference).item()})
    assert bits(default) != bits(original)

{'dtype': 'float32', 'original': 1.5241557359695435, 'JAX': 1.5241557914777246, 'reference': 1.5241557359695435}
{'dtype': 'complex64', 'original': np.complex128(0j), 'JAX': (4.9999999981995875e-46+0j), 'reference': 0j}


## Tiny complex division and local bins

Squaring components of order 1e−23 underflows in float32; a complex quotient avoids that unnecessary denominator. The installed NumPy complex division rounds a reciprocal scale before multiplying its components; the explicit control reproduces its single-precision quotient, which is one ULP below the observed JAX result here. This describes these inputs and builds, without asserting a universal JAX lowering. See the [NumPy complex division source](https://github.com/numpy/numpy/blob/v2.5.2/numpy/_core/src/umath/loops.c.src#L2032-L2060). Separately, subtracting large global prefix sums can erase a small later bin. Local bin reductions follow the original summary formula.

In [3]:
tiny = np.asarray([1e-23+2e-23j], np.complex64)
with np.errstate(divide="ignore", invalid="ignore", under="ignore"):
    squared = tiny.real*tiny.real + tiny.imag*tiny.imag
    unsafe = squared/squared
with JAXScheme():
    value = jnp.asarray(tiny)
    real, imag = relbin_jax._cdiv(value.real, value.imag, value.real, value.imag)
    safe = np.asarray(real+1j*imag)
assert np.isnan(unsafe).all()
np.testing.assert_allclose(safe, [1.+0j], rtol=1e-7)
with JAXScheme(reference_operations=("divide",)):
    real, imag = relbin_jax._cdiv(value.real, value.imag, value.real, value.imag)
    native_quotient = real+1j*imag
original_quotient = np.divide(tiny, tiny)
exact(native_quotient, original_quotient)
ratio = np.asarray(tiny.real/tiny.imag, dtype=np.float32)
scale = np.asarray(1./(tiny.real*ratio+tiny.imag), dtype=np.float32)
control_real = (tiny.real*ratio+tiny.imag)*scale
np.testing.assert_array_equal(control_real, original_quotient.real)
print({"squared denominator": squared.tolist(), "safe JAX quotient": safe.tolist(),
       "original rounded-reciprocal quotient": original_quotient.tolist(), "divide control exact": True})

values = np.asarray([1e16+0j, 1+0j], np.complex128)
bins = np.asarray([[1, 2]], np.int64)
freqs = np.asarray([0., 1.])
psd = np.ones(2)
reference = np.ones(2, np.complex128)
from types import SimpleNamespace
from pycbc.inference.models.relbin import Relative
state = SimpleNamespace(psds={"D": psd}, df={"D": 1.}, f={"D": freqs})
original = Relative.summary_product(state, reference, values, bins, "D")
with JAXScheme():
    args = (jnp.asarray(reference), jnp.asarray(values), psd, freqs, bins, 1.)
    default = relbin_jax.summary_product(*args)
with JAXScheme(reference_operations=("relbin_summary",)):
    native = relbin_jax.summary_product(*args)
for a, b in zip(native, original): exact(a, b)
for a, b in zip(default, original): exact(a, b)
prefix = np.cumsum(values)
print({"global prefix difference": (prefix[1]-prefix[0]).real,
       "original/local a0": np.asarray(default[0]).real.tolist()})
assert prefix[1]-prefix[0] == 0 and default[0][0] == 4

{'squared denominator': [0.0], 'safe JAX quotient': [(1+0j)], 'original rounded-reciprocal quotient': [(0.9999999403953552+0j)], 'divide control exact': True}


{'global prefix difference': np.float64(0.0), 'original/local a0': [4.0]}


## Phase and GPS conventions

The original compiled relative kernel uses 3.141592653. Keeping that constant preserves its phase convention; using full pi is a distinct mathematical input. Default detector projection centers a GPS timestamp before adding its small delay. The original adds the delay to absolute GPS first, which rounds before subtracting the epoch. The time_shift control restores that ordering.

In [4]:
frequencies = np.asarray([20., 500.])
delay = .007123456789
ones = np.ones(2, np.complex128)
# a0=0, a1=1 returns the conjugated difference of the two shifted edges.
original_phase_difference = relbin_cpu.likelihood_parts_det(
    frequencies, delay, ones, ones, np.zeros(1, np.complex128),
    np.ones(1, np.complex128), np.zeros(1), np.zeros(1))[0]
with JAXScheme():
    default = relbin_jax.likelihood_parts_det(
        jnp.asarray(frequencies), delay, jnp.asarray(ones), jnp.asarray(ones),
        jnp.zeros(1, jnp.complex128), jnp.ones(1, jnp.complex128),
        jnp.zeros(1), jnp.zeros(1))[0]
with JAXScheme(reference_operations=("relbin_likelihood",)):
    native = relbin_jax.likelihood_parts_det(
        jnp.asarray(frequencies), delay, jnp.asarray(ones), jnp.asarray(ones),
        jnp.zeros(1, jnp.complex128), jnp.ones(1, jnp.complex128),
        jnp.zeros(1), jnp.zeros(1))[0]
exact(native, original_phase_difference)
full_pi = np.conjugate(np.diff(np.exp(-2j*np.pi*delay*frequencies))[0])
print({"compiled constant residual": float(abs(default-original_phase_difference)),
       "full pi residual": float(abs(full_pi-original_phase_difference))})
assert abs(full_pi-original_phase_difference) > 1e-9

reference_time, epoch, offset = 1126259460.2, 1126259460., .007123456789
original_dt = (reference_time+offset)-epoch
with JAXScheme():
    _, centered_dt = _arrival_time_and_shift(reference_time, jnp.asarray(offset), epoch)
with JAXScheme(reference_operations=("time_shift",)):
    _, native_dt = _arrival_time_and_shift(reference_time, jnp.asarray(offset), epoch)
exact(native_dt, original_dt)
print({"original relative seconds": original_dt, "centered seconds": float(centered_dt),
       "difference seconds": float(centered_dt-original_dt)})
assert float(centered_dt) != original_dt

{'compiled constant residual': 0.0, 'full pi residual': 4.349023916834699e-09}
{'original relative seconds': 0.2071235179901123, 'centered seconds': 0.20712350447271582, 'difference seconds': -1.3517396479922894e-08}


## Marginalization and gradients

The phase likelihood uses the scaled Bessel function; vector marginalization uses weighted log-sum-exp. JAX and SciPy can round differently even for identical formulas. The whole original marginalization is independently selectable. This gradient check covers the device numerical kernel, not a sampler, host random draw, or reference route.

In [5]:
sh = np.asarray([1.35+.21j, -2.01+.3j, .75-.35j])
hh = np.asarray([1.05, 3.11, .69])
weights = np.log(np.asarray([.2, .3, .5]))
original = tools.marginalize_likelihood(sh, hh, phase=True, logw=weights)
with JAXScheme():
    default = tools.marginalize_likelihood(jnp.asarray(sh), jnp.asarray(hh), phase=True, logw=weights)
with JAXScheme(reference_operations=("inference_marginalization",)):
    native = tools.marginalize_likelihood(jnp.asarray(sh), jnp.asarray(hh), phase=True, logw=weights)
exact(native, original)
print({"original": original, "JAX": default, "residual": default-original})
with JAXScheme():
    data = jnp.asarray([1.+.5j, -.2+.3j])
    def loss(amplitude):
        hd, norm = tools_jax.fused_inner_hd_hh(amplitude*data, data)
        return tools.marginalize_likelihood(hd, norm, phase=True, skip_vector=True)
    gradient = jax.jit(jax.grad(loss))(jnp.asarray(.7))
    step = 1e-5
    finite = (loss(.7+step)-loss(.7-step))/(2*step)
    np.testing.assert_allclose(gradient, finite, rtol=1e-8)
    print({"gradient": float(gradient), "finite difference": float(finite)})

{'original': -0.29379183166501655, 'JAX': -0.29379183166501655, 'residual': 0.0}
{'gradient': -0.3667865554017467, 'finite difference': -0.3667865554196625}


## Distance-grid interpolation

The distance grid is prepared by the original SciPy setup. JAX evaluates its knots and coefficients with device B-spline basis arithmetic; the original interpolation control independently restores SciPy evaluation. This example reports the observed residual without assuming the libraries use the same instruction ordering. The device evaluator also supports a derivative through its query coordinates.

In [6]:
from scipy.interpolate import RectBivariateSpline
x = np.linspace(.1, 5., 6)
y = np.linspace(.2, 4., 6)
spline = RectBivariateSpline(x, y, np.outer(x, y))
evaluate = tools_jax.rect_bivariate_spline_evaluator(spline)
query_x = np.asarray([.5, 1.25, 2.5])
query_y = np.asarray([.8, 1., 2.])
original = spline(query_x, query_y, grid=False)
with JAXScheme():
    default = evaluate(jnp.asarray(query_x), jnp.asarray(query_y))
    derivative = jax.grad(lambda z: evaluate(z, jnp.asarray(.8)))(jnp.asarray(.7))
    np.testing.assert_allclose(derivative, .8, rtol=1e-12)
with JAXScheme(reference_operations=("inference_interpolant",)):
    native = evaluate(jnp.asarray(query_x), jnp.asarray(query_y))
exact(native, original)
print({"maximum spline residual": float(np.max(np.abs(default-original))),
       "original interpolation exact": True, "coordinate derivative": float(derivative)})

{'maximum spline residual': 2.220446049250313e-16, 'original interpolation exact': True, 'coordinate derivative': 0.8}


## CDF boundaries and the RNG stream

All paths below use the same NumPy uniform stream. Exponentiation, cumulative sums and normalization can move a boundary enough to select a different index. The reference switch calls the original weighted draw and preserves the following RNG value as well.

In [7]:
logs = np.random.default_rng(11).normal(size=4096).astype(np.float32)
state = np.random.get_state()
try:
    np.random.seed(419)
    original = tools.draw_sample(logs, size=100000)
    next_original = np.random.random()
    np.random.seed(419)
    with JAXScheme():
        default = tools_jax.draw_sample(jnp.asarray(logs), size=100000)
    np.random.seed(419)
    with JAXScheme(reference_operations=("inference_sampling",)):
        native = tools_jax.draw_sample(jnp.asarray(logs), size=100000, host=False)
    next_native = np.random.random()
    exact(native, original)
    assert next_native == next_original
    print({"different default indices": int(np.count_nonzero(default != original)),
           "reference exact": True, "following RNG value exact": True})
finally:
    np.random.set_state(state)

{'different default indices': 41, 'reference exact': True, 'following RNG value exact': True}


## Dominant-mode SNR normalization

The compiled predictor and the operations after it are separate boundaries. Identical kernel outputs isolate a division residual: in this build, multiplying the magnitude by a rounded reciprocal reproduces default JAX, while the original divides directly. Selecting `relbin_snr_normalization` restores that step. Holding the predictor and proposed parameters fixed also isolates the final complex likelihood product; `relbin_likelihood` restores that original combination.

In [8]:
from pycbc.inference.models.relbin import RelativeTimeDom
from pycbc.inference.models.relative_jax import JAXRelativeTimeDom
rng = np.random.default_rng(205)
wave = lambda: rng.normal(size=3)+1j*rng.normal(size=3)
values = dict(hp=wave(), hc=wave(), h00=wave(), a0=wave()[:2], a1=wave()[:2],
              b0=np.ones(2), b1=np.full(2, .1))
results = []
intermediates = []
likelihoods = []
for context in (CPUScheme(), JAXScheme(reference_operations=("relbin_snr", "inference_projection", "inference_time_interpolation")),
                JAXScheme(reference_operations=("relbin_snr", "relbin_snr_normalization", "inference_projection", "inference_time_interpolation", "relbin_likelihood"))):
    with context:
        device = isinstance(context, JAXScheme)
        model = object.__new__(JAXRelativeTimeDom if device else RelativeTimeDom)
        model.sample_rate = 1000.
        model.tstart = {"H1": -.01}; model.end_time = {"H1": 0.}; model.ta = {"H1": 0.}
        model.num_samples = {"H1": 8}
        model.fedges = {"H1": np.asarray([20., 80., 200.])}
        model.h00_sparse = {"H1": values["h00"]}
        model.sdat = {"H1": {key: values[key] for key in ("a0", "a1", "b0", "b1")}}
        if device:
            model._get_jax_likelihood_data = lambda ifo, hp: tuple(jnp.asarray(v) for v in (
                model.fedges[ifo], model.h00_sparse[ifo], values["a0"], values["a1"], values["b0"], values["b1"]))
        wfs = {"H1": tuple(jnp.asarray(values[key]) if device else values[key] for key in ("hp", "hc"))}
        results.append(model.get_snr(wfs)["H1"].numpy())
        intermediates.append((model.sh["H1"].numpy(), np.asarray(model.hh["H1"])))
        model._current_params = dict(inclination=.41, polarization=.31, tc=-.007, ra=1., dec=.2)
        model.marginalize_vector_params = {}; model.vsamples = 1
        model.get_waveforms = lambda *args, **kwargs: wfs
        # The predictor has already run: hold its outputs and proposal fixed.
        model.get_snr = lambda wfs: {}
        model.snr_draw = lambda **kwargs: None
        model.precalc_antenna_factors = True
        model.get_precalc_antenna_factors = lambda ifo: (.35687123, .673249, 0.)
        model.return_sh_hh = True
        model.marginalize_loglr = lambda sh, hh: np.real(sh)-.5*hh
        model._data = {"H1": None}; model._static_params = {}; model.waveform_transforms = None
        likelihoods.append(model._loglr())
exact(results[2], results[0])
for samples, norm in intermediates[1:]:
    exact(samples, intermediates[0][0]); exact(norm, intermediates[0][1])
samples, norm = intermediates[0]
with JAXScheme():
    magnitude = np.asarray(jnp.abs(jnp.asarray(samples[2:-2])))
    root = np.asarray(jnp.asarray(norm)**.5)
print({"kernel outputs exact": True,
       "magnitude residual": float(np.max(np.abs(magnitude-np.abs(samples[2:-2])))),
       "scalar root residual": float(root-float(norm)**.5),
       "normalized residual": float(np.max(np.abs(results[1]-results[0]))),
       "normalization reference exact": True})
reciprocal_control = np.abs(samples[2:-2])*np.float64(1.0/(float(norm)**.5))
exact(results[1], reciprocal_control)
assert np.count_nonzero(reciprocal_control != results[0]) > 0
for actual, expected in zip(likelihoods[2], likelihoods[0]):
    exact(np.asarray(actual), np.asarray(expected))
print({"rounded reciprocal reproduces default": True,
       "post-kernel likelihood residual": float(abs(likelihoods[1][0]-likelihoods[0][0])),
       "post-kernel likelihood reference exact": True})


{'kernel outputs exact': True, 'magnitude residual': 0.0, 'scalar root residual': 0.0, 'normalized residual': 4.440892098500626e-16, 'normalization reference exact': True}
{'rounded reciprocal reproduces default': True, 'post-kernel likelihood residual': 1.1102230246251565e-16, 'post-kernel likelihood reference exact': True}


## Polarization and reconstruction preparation

The rotation uses a complex multiply in NumPy and separate real/imaginary expressions in JAX. Fixed phase inputs isolate the product rounding without attributing it to an uninspected machine instruction. Inclination and phase residuals are reported separately. Reconstruction also has a float64 phase grid and a log-weight preparation boundary; the sampling control restores these before the original weighted draw.

In [9]:
rng = np.random.default_rng(81)
fp, fc = rng.normal(size=(2, 100))
pol = rng.uniform(0., 2*np.pi, 100)
inc = rng.uniform(0., np.pi, 100)
phase = np.exp(-2j*pol)
rotated = (fp+1j*fc)*phase
cosine = np.cos(inc)
original = rotated.real*(.5*(1+cosine*cosine)) + 1j*rotated.imag*cosine
with JAXScheme():
    like = jnp.ones(100, jnp.complex128)
    default = np.asarray(relbin_jax.dominant_mode_projection(fp, fc, pol, inc, like))
    device_phase = np.asarray(relbin_jax.polarization_phase(pol, like))
    device_cosine = np.asarray(jnp.cos(jnp.asarray(inc)))
    real, imag = relbin_jax.polarized_antenna_response(fp, fc, phase, like)
    device_rotated = np.asarray(real+1j*imag)
with JAXScheme(reference_operations=("inference_projection",)):
    restored = relbin_jax.dominant_mode_projection(fp, fc, pol, inc, like)
exact(restored, original)
print({"phase residual": float(np.max(abs(device_phase-phase))),
       "inclination cosine residual": float(np.max(abs(device_cosine-cosine))),
       "rotation residual with fixed phase": float(np.max(abs(device_rotated-rotated))),
       "complete projection residual": float(np.max(abs(default-original))),
       "projection reference exact": True})

# Phase reconstruction uses the original float64 grid, including for complex64 inputs.
sh = np.complex64(1.234567+.4321j); hh = np.float32(.72)
phase_grid = np.linspace(0, 2*np.pi, 17)
expected = (np.exp(-2j*phase_grid)*sh).real+hh
with JAXScheme():
    grid, phase_values = tools_jax.phase_reconstruction_values(jnp.asarray(sh), jnp.asarray(hh), 17)
    assert grid.dtype == phase_grid.dtype
with JAXScheme(reference_operations=("inference_sampling",)):
    native_grid, native_values = tools_jax.phase_reconstruction_values(jnp.asarray(sh), jnp.asarray(hh), 17)
exact(native_grid, phase_grid); exact(native_values, expected)
rng = np.random.default_rng(881)
values = rng.normal(size=100).astype(np.float32)
weights = rng.uniform(.1, 1., 100).astype(np.float32)
with JAXScheme():
    device_weights = np.asarray(tools_jax.weighted_loglr(jnp.asarray(values), weights))
with JAXScheme(reference_operations=("inference_sampling",)):
    native_weights = tools_jax.weighted_loglr(jnp.asarray(values), weights)
exact(native_weights, values+np.log(weights))
print({"phase reconstruction residual": float(np.max(abs(np.asarray(phase_values)-expected))),
       "log-weight residual": float(np.max(abs(device_weights-(values+np.log(weights))))),
       "sampling preparation reference exact": True})

{'phase residual': 0.0, 'inclination cosine residual': 0.0, 'rotation residual with fixed phase': 4.47545209131181e-16, 'complete projection residual': 3.1401849173675503e-16, 'projection reference exact': True}
{'phase reconstruction residual': 1.1102230246251565e-16, 'log-weight residual': 1.1920928955078125e-07, 'sampling preparation reference exact': True}


## Physical model composition

The data amplitude is 1e−23, the PSD is 1e−46, and the binary has masses 30/20 solar masses at 500 Mpc. Every model is rebuilt in its context. The independent waveform, projection, weighting, inner, timing, filtering, detector and marginalization controls compose to the original result. NumPy FFT is selected explicitly for deterministic original FFT comparisons; the configuration is restored afterward.

In [10]:
import logging
from pycbc.fft import backend_cpu
from pycbc.inference.models.gaussian_noise import GaussianNoise
from pycbc.inference.models.relbin import Relative
from pycbc.inference.models.marginalized_gaussian_noise import MarginalizedTime, MarginalizedPolarization
controls = ("waveform", "time_shift", "inference_whitening", "inference_inner",
            "inference_projection", "inference_time_interpolation", "inference_marginalization",
            "correlate", "divide", "ifft", "fft", "squared_norm", "inner",
            "detector.Detector.antenna_pattern", "detector.Detector.arrival_time",
            "detector.Detector.time_delay_from_location",
            "relbin_summary", "relbin_likelihood", "relbin_snr")
static = dict(approximant="TaylorF2", mass1=30., mass2=20., distance=500., inclination=.4,
              f_lower=20., coa_phase=.2, tc=1126259460.2, ra=1.1, dec=-.3)
old_fft = backend_cpu.cpu_backend
logger = logging.getLogger()
old_level = logger.level
logger.setLevel(logging.ERROR)
state = np.random.get_state()
try:
    backend_cpu.set_backend(["numpy"])
    for model_type in (GaussianNoise, MarginalizedTime, MarginalizedPolarization, Relative):
        results = []
        for context in (CPUScheme(), JAXScheme(), JAXScheme(reference_operations=controls)):
            with context:
                data = FrequencySeries(np.full(129, 1e-23+1e-23j, np.complex128),
                                       delta_f=2., epoch=1126259460.)
                psd = FrequencySeries(np.full(129, 1e-46), delta_f=2.)
                model = model_type(("polarization",), {"H1": data}, {"H1": 20.},
                                   psds={"H1": psd}, static_params=static,
                                   **({"fiducial_params": dict(static, polarization=.2)} if model_type is Relative else {}))
                model.update(polarization=.2 if model_type in (GaussianNoise, Relative) else np.linspace(0., 2*np.pi, 8))
                results.append(np.asarray(model.loglr))
        exact(results[2], results[0])
        assert np.isfinite(results[1])
        print({"model": model_type.name, "original": results[0].item(),
               "default JAX": results[1].item(), "all controls exact": True})
finally:
    backend_cpu.cpu_backend = old_fft
    logger.setLevel(old_level)
    np.random.set_state(state)

{'model': 'gaussian_noise', 'original': -343.5158103466887, 'default JAX': -343.51628687978365, 'all controls exact': True}


{'model': 'marginalized_time', 'original': -149.18652918247454, 'default JAX': -149.18652918247295, 'all controls exact': True}


{'model': 'marginalized_polarization', 'original': -149.09112600656337, 'default JAX': -149.0911260065621, 'all controls exact': True}


{'model': 'relative', 'original': -162.55256942326957, 'default JAX': -162.5525694232682, 'all controls exact': True}
